# ABC2027 UWB — starter

Run this notebook from the unpacked package. It loads validation stream `val_01`, plots the annotated falls, builds one extra short stream from the included segments, then runs the energy baseline and scores it locally.

In [ ]:
import csv
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import h5py
import matplotlib.pyplot as plt
import numpy as np

def package_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "streams_events.csv").is_file() and (candidate / "src" / "abc2027").is_dir():
            return candidate
    raise FileNotFoundError("ABC2027 package root not found from the working directory")

ROOT = package_root()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

import build_streams
from abc2027.baseline import STEP_S, energy_events, events_from_windows, fit_window_model, iter_windows, labels_for_windows, window_features
from abc2027.evaluate import evaluate, load_ground_truth, validate_predictions
from abc2027.streams import Segment

STREAM_ID = "val_01"
STREAM_PATH = ROOT / "streams" / "val" / "stream_01.h5"
EVENTS_PATH = ROOT / "streams_events.csv"
COMPOSITION_PATH = ROOT / "streams_composition.csv"

## Load the validation stream and its falls

In [ ]:
events = [
    row for row in csv.DictReader(EVENTS_PATH.open(encoding="utf-8"))
    if row["stream_id"] == STREAM_ID
]
assert len(events) >= 3
with h5py.File(STREAM_PATH, "r") as handle:
    cir = handle["cir"][:]
    t = handle["t"][:]
print(STREAM_ID, cir.shape, "falls", len(events))

## Plot energy with every annotated fall on this stream

In [ ]:
energy = np.mean(np.sum(cir.astype(np.float64) ** 2, axis=2), axis=0)
figure_dir = ROOT / "outputs"
figure_dir.mkdir(parents=True, exist_ok=True)
figure, axis = plt.subplots(figsize=(10, 3))
axis.plot(t[::10], energy[::10], color="black", linewidth=0.6)
for row in events:
    axis.axvspan(float(row["t_start"]), float(row["t_end"]), color="tab:red", alpha=0.35)
axis.set_xlabel("t (s)")
axis.set_ylabel("mean ||CIR||^2")
axis.set_title(STREAM_ID)
figure.tight_layout()
figure.savefig(figure_dir / "val_example.png", dpi=120)
plt.close(figure)
print("plotted", len(events), "falls")

## Generate one extra short stream

Official streams stay unchanged. Two segments already stored in this package are joined with the same 0.2 s crossfade as the published streams.

In [ ]:
composition = list(csv.DictReader(COMPOSITION_PATH.open(encoding="utf-8")))
fall_row = next(row for row in composition if row["stream_id"] == STREAM_ID and row["kind"] == "fall")
adl_row = next(row for row in composition if row["stream_id"] == STREAM_ID and row["kind"] == "adl")
event_row = next(row for row in events if row["event_id"] == fall_row["segment_id"])

def segment_from(row, event=None):
    return Segment(
        kind=row["kind"],
        path=row["path"],
        label=row["label"],
        dataset=row["dataset"],
        source_t0=float(row["source_t0"]),
        source_t1=float(row["source_t1"]),
        event_t0=None if event is None else float(event["source_t_start"]),
        event_t1=None if event is None else float(event["source_t_end"]),
        fall_type="" if event is None else event["fall_type"],
        quality="" if event is None else event["quality"],
        needs_review=False if event is None else event["needs_review"].lower() == "true",
        ignore=False if event is None else event["ignore"].lower() == "true",
    )

extra = ROOT / "outputs" / "starter_extra.h5"
extra_events, extra_parts = build_streams._write_stream(
    extra, "val", 99, [segment_from(adl_row), segment_from(fall_row, event_row)]
)
with h5py.File(extra, "r") as handle:
    print("extra", tuple(int(value) for value in handle["cir"].shape), "events", len(extra_events), "segments", len(extra_parts))

## Baselines on this validation stream

The energy threshold is the one tuned on all of validation. The logistic model is fit here on `train_00` only, so this cell demonstrates the code. The published validation score uses every training stream.

In [ ]:
detected = energy_events(cir, percentile=98.0, min_duration=0.6, merge_gap=0.8)
submission = ROOT / "outputs" / "starter_submission.csv"
with submission.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["stream_id", "t_start", "t_end", "score", "t_alarm"])
    writer.writeheader()
    for event in detected:
        writer.writerow({
            "stream_id": STREAM_ID,
            "t_start": f"{event['t_start']:.3f}",
            "t_end": f"{event['t_end']:.3f}",
            "score": f"{event['score']:.6f}",
            "t_alarm": f"{event['t_alarm']:.3f}",
        })
duration = cir.shape[1] / 50.0
rows = validate_predictions(submission, {STREAM_ID: duration})
ground_path = ROOT / "outputs" / "starter_events.csv"
with ground_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(events[0]))
    writer.writeheader()
    writer.writerows(events)
local = evaluate(load_ground_truth(ground_path, ignore_flagged=True), rows, {STREAM_ID: duration})
print("energy predictions", len(rows), "f1_mean", local["ranking_score"]["f1_mean"])
assert len(rows) > 0 and local["ranking_score"]["f1_mean"] > 0.0

def window_table(path, stream_events):
    with h5py.File(path, "r") as handle:
        block = handle["cir"][:]
    spans = iter_windows(block.shape[1])
    features = np.stack([window_features(block[:, start:stop, :]) for start, stop in spans])
    times = [(start * STEP_S, stop * STEP_S) for start, stop in spans]
    labelled = [(float(row["t_start"]), float(row["t_end"])) for row in stream_events]
    return features, labels_for_windows(times, labelled), times

train_events = [
    row for row in csv.DictReader(EVENTS_PATH.open(encoding="utf-8"))
    if row["stream_id"] == "train_00" and row["ignore"].lower() != "true"
]
train_x, train_y, _times = window_table(ROOT / "streams" / "train" / "stream_00.h5", train_events)
model = fit_window_model(train_x, train_y)
_features, _labels, times = window_table(STREAM_PATH, events)
scores = model.predict_proba(_features)[:, 1]
logistic_events = events_from_windows(times, scores, 0.85)
print("logistic predictions", len(logistic_events))
